In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import pyarrow as pa
import pyarrow.parquet as pq
import re
import os
from difflib import get_close_matches

In [2]:
# Importar RIPS
os.listdir('/kaggle/input/datasets/geraldinelaverde/rips-inicial')
ruta_archivo = '/kaggle/input/datasets/geraldinelaverde/rips-inicial/RIPS_0.parquet'
df_rips = pd.read_parquet(ruta_archivo)
df_rips.head()

,Departamento,Municipio,Año,TipoAtencion,Diagnostico,NumeroAtenciones
0,05 - Antioquia,05212 - Copacabana,2012,CONSULTAS,F431 - TRASTORNO DE ESTRÃ‰S POSTRAUMATICO,2
1,05 - Antioquia,05576 - Pueblorrico,2011,CONSULTAS,"C779 - TUMOR MALIGNO DEL GANGLIO LINFATICO, SI...",1
2,05 - Antioquia,05086 - Belmira,2014,PROCEDIMIENTOS DE SALUD,Z340 - SUPERVISION DE PRIMER EMBARAZO NORMAL,48
3,05 - Antioquia,05490 - NecoclÃ­,2015,CONSULTAS,M545 - LUMBAGO NO ESPECIFICADO,961
4,05 - Antioquia,05212 - Copacabana,2012,CONSULTAS,F510 - INSOMNIO NO ORGANICO,32


In [3]:
#Análisis Exploratorio

print("🔹 SHAPE")
print(df_rips.shape)

print("\n🔹 HEAD")
print(df_rips.head())

print("\n🔹 INFO")
print(df_rips.info())

print("\n🔹 NULOS POR COLUMNA")
print(df_rips.isna().sum().sort_values(ascending=False))

print("\n🔹 TIPOS DE DATOS")
print(df_rips.dtypes)

print("\n🔹 DESCRIPTIVOS NumeroAtenciones")
print(df_rips['NumeroAtenciones'].describe())

# ------------------------------
# Valores únicos
# ------------------------------
print("\n🔹 VALORES ÚNICOS")
for col in ['Departamento','Municipio','TipoAtencion','Diagnostico','Año']:
    print(f'{col}:', df_rips[col].nunique())

# ------------------------------
# Top diagnósticos
# ------------------------------
print("\n🔹 TOP 10 DIAGNÓSTICOS")
print(
    df_rips.groupby('Diagnostico')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
           .head(10)
)

# ------------------------------
# Atenciones por departamento
# ------------------------------
print("\n🔹 ATENCIONES POR DEPARTAMENTO")
print(
    df_rips.groupby('Departamento')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
)

# ------------------------------
# Tipo de atención
# ------------------------------
print("\n🔹 ATENCIONES POR TIPO")
print(
    df_rips.groupby('TipoAtencion')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
)

# ------------------------------
# Municipios con más carga
# ------------------------------
print("\n🔹 TOP 15 MUNICIPIOS")
print(
    df_rips.groupby('Municipio')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
           .head(15)
)

🔹 SHAPE
(38000000, 6)

🔹 HEAD
     Departamento            Municipio   Año             TipoAtencion  \
0  05 - Antioquia   05212 - Copacabana  2012                CONSULTAS   
1  05 - Antioquia  05576 - Pueblorrico  2011                CONSULTAS   
2  05 - Antioquia      05086 - Belmira  2014  PROCEDIMIENTOS DE SALUD   
3  05 - Antioquia     05490 - NecoclÃ­  2015                CONSULTAS   
4  05 - Antioquia   05212 - Copacabana  2012                CONSULTAS   

                                         Diagnostico  NumeroAtenciones  
0          F431 - TRASTORNO DE ESTRÃ‰S POSTRAUMATICO                 2  
1  C779 - TUMOR MALIGNO DEL GANGLIO LINFATICO, SI...                 1  
2       Z340 - SUPERVISION DE PRIMER EMBARAZO NORMAL                48  
3                     M545 - LUMBAGO NO ESPECIFICADO               961  
4                        F510 - INSOMNIO NO ORGANICO                32  

🔹 INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38000000 entries, 0 to 37999999
Dat

In [4]:
#Limpieza Inicial

# =========================================
# LIMPIEZA ESTRUCTURAL df_rips
# =========================================

import pandas as pd

# 1) Corregir codificación de textos
def fix_encoding(text):
    try:
        return text.encode('latin1').decode('utf-8')
    except:
        return text

for col in ['Departamento','Municipio','TipoAtencion','Diagnostico']:
    df_rips[col] = df_rips[col].astype(str).apply(fix_encoding)

# 2) Quitar basura que se metió como filas
df_rips = df_rips[~df_rips['Departamento'].str.contains('message|error|status|{|}', regex=True)]

# 3) Corregir tipos de datos
df_rips['Año'] = df_rips['Año'].astype('Int64')
df_rips['NumeroAtenciones'] = df_rips['NumeroAtenciones'].astype('Int64')

# 4) Eliminar nulos reales
df_rips = df_rips.dropna()

# 5) Reset index
df_rips = df_rips.reset_index(drop=True)

print("✅ Limpieza estructural terminada")
print(df_rips.info())
print(df_rips.head())

# Guardar dataframe como parquet
df_rips.to_parquet('/kaggle/working/df_rips_limpio.parquet', index=False)

✅ Limpieza estructural terminada
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38000000 entries, 0 to 37999999
Data columns (total 6 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   Departamento      object
 1   Municipio         object
 2   Año               Int64 
 3   TipoAtencion      object
 4   Diagnostico       object
 5   NumeroAtenciones  Int64 
dtypes: Int64(2), object(4)
memory usage: 1.8+ GB
None
     Departamento            Municipio   Año             TipoAtencion  \
0  05 - Antioquia   05212 - Copacabana  2012                CONSULTAS   
1  05 - Antioquia  05576 - Pueblorrico  2011                CONSULTAS   
2  05 - Antioquia      05086 - Belmira  2014  PROCEDIMIENTOS DE SALUD   
3  05 - Antioquia      05490 - Necoclí  2015                CONSULTAS   
4  05 - Antioquia   05212 - Copacabana  2012                CONSULTAS   

                                         Diagnostico  NumeroAtenciones  
0          F431 - TRASTORNO DE ESTRÃ‰S POSTRAU